# simple_agents_framework on Colab

Markdown file in, callable agent out. Run the cells top to bottom.

Repo: https://github.com/Agentsable/simple_agents_framework


## 1. Install

The framework is a pip install; the Claude Agent SDK shells out to the
`claude` CLI, so that gets installed too (npm, ~30s).


In [ ]:
!pip install -q git+https://github.com/Agentsable/simple_agents_framework.git
!npm install -g @anthropic-ai/claude-code >/dev/null 2>&1
!claude --version


## 2. Mount Drive

Your files land under `/content/drive/MyDrive`. Point the agent at a folder
there and it can read them.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORKDIR = '/content/drive/MyDrive'  # the folder the agent works in
!ls {WORKDIR} | head


## 3. Anthropic API key

Best: put it in Colab **Secrets** (🔑 in the left sidebar) as
`ANTHROPIC_API_KEY` and flick on notebook access — then this cell just picks
it up. Otherwise it prompts, hidden, and the key never lands in the
notebook file.


In [ ]:
import getpass

try:
    from google.colab import userdata
    API_KEY = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    API_KEY = getpass.getpass('Anthropic API key: ')

assert API_KEY.startswith('sk-'), 'that does not look like an API key'
print('key loaded:', API_KEY[:7] + '…')


## 4. Define an agent

Frontmatter configures it, the body is the system prompt. Every key is
optional — see the README for the full list.


In [ ]:
from pathlib import Path

Path('doc_reader.md').write_text('''---
name: doc_reader
description: Reads the documents in a folder and answers questions about them
model: claude-sonnet-5
tools: Read, Grep, Glob
---
You answer questions about the files in this folder.

Rules:
- Find the relevant files (Glob/Grep) and Read them before answering. Never guess.
- Answer in at most 5 bullets, each one sentence.
- Cite locations as `file:line`.
- If the files are missing or contradict each other, say so plainly.
''')


## 5. Ask it something

`stream=html_stream` renders the run live: blue sent, green agent, slate
thinking, amber tool call, cyan tool result, red error. Drop `stream=` to
just get the string back.


In [ ]:
import simple_agents_framework as saf
from plugins.streaming.jupyter_html import html_stream

agent = saf.create_agent_from_markdown('doc_reader.md', API_KEY, cwd=WORKDIR)
agent.ask('What files are in this folder, and what is each one for?',
          stream=html_stream)


Follow-ups resume the same session, so context carries over.


In [ ]:
agent.ask('Which of those changed most recently?', stream=html_stream)
